In [ ]:
import numpy as np
import pandas as pd
from statsmodels.api import Logit 
import statsmodels.api as sm
import os
import s3fs
import sys
sys.path.append('../outils')

import fonctions_nomenclature as fnom
import fonctions_preparation_table as fprep
%load_ext autoreload
%autoreload 2

np.random.seed(123)

# Create filesystem object
S3_ENDPOINT_URL = "https://" + os.environ["AWS_S3_ENDPOINT"]
fs = s3fs.S3FileSystem(client_kwargs={'endpoint_url': S3_ENDPOINT_URL})
BUCKET = "mathildedonnenfeld/diffusion/bistrots_et_bulletins"


FILE_KEY_S3 = "table_analyse_muni.csv"
FILE_PATH_S3 = BUCKET + "/" + FILE_KEY_S3

with fs.open(FILE_PATH_S3, mode="rb") as file_in:
    df = pd.read_csv(file_in, sep=",")

df.head()   

In [ ]:
# Si la lecture du dataframe n'a pas fonctionné par minio
if ('df' not in globals()):
    df = pd.read_pickle(
    "/home/onyxia/work/bistrots-et-bulletins/data_load_and_prep/table_analyse_muni.pkl"
)

In [ ]:
FILE_KEY_S3 = "table_analyse_muni_sous_ech.csv"
FILE_PATH_S3 = BUCKET + "/" + FILE_KEY_S3

with fs.open(FILE_PATH_S3, mode="rb") as file_in:
    df_sous_ech = pd.read_csv(file_in, sep=",")


In [ ]:
# Si la lecture du dataframe n'a pas fonctionné par minio
if ('df_sous_ech' not in globals()):
    df = pd.read_pickle(
    "/home/onyxia/work/bistrots-et-bulletins/data_load_and_prep/df_sous_ech.pkl"
)

# Préparation des données pour la régression 

In [ ]:
df_sous_ech.shape[0]

In [ ]:
# Compter le nombre de valeurs manquantes dans la table finale 

colonnes = [
    'POPULATION',
    'NIVVIE_MEDIAN',
    'TAUX_PAUVRETE',
    'nb_etablissements',
    'age_moyen_bistrots',
    'Nombre candidats',
    'Abstentions'
]

print(df_sous_ech[colonnes].isna().sum())
print(df_sous_ech[df_sous_ech[colonnes].isna().any(axis=1)])

In [ ]:
# Supprimer les valeurs manquantes : 
df_sous_ech = df_sous_ech.dropna(subset=['NIVVIE_MEDIAN']) # certaines communes ne sont pas diffusées dans filosofi par mesure de secret stat
df_sous_ech = df_sous_ech.dropna(subset=['TAUX_PAUVRETE']) # idem 
#df_sous_ech = df_sous_ech.dropna(subset=['age_moyen_bistrots'])  on garde car c'est pour les villes n'ayant pas de bistrot

In [ ]:
# Préparation des variables 
df_sous_ech['ind1']= df_sous_ech['ind1'].astype("category")
df_sous_ech['ind2']= df_sous_ech['ind2'].astype("category")
df_sous_ech['ind3']= df_sous_ech['ind3'].astype("category")
df_sous_ech['ind4']= df_sous_ech['ind4'].astype("category")
df_sous_ech["bistrot_dens"]=df_sous_ech['nb_etablissements']/df_sous_ech['POPULATION']*1000
df_sous_ech["tx_abs"] = df_sous_ech['Abstentions']/(df_sous_ech['Inscrits'])

# Pré traitement de l'âge des bistrots pour lesquels NaN car pas de bistrot : 
# on met age = 0, ce qui a un sens par rapport à la variable (pas de bistro, donc )

df_sous_ech['age_moyen_bistrots'] = df_sous_ech['age_moyen_bistrots'].fillna(0)

In [ ]:
df_sous_ech.describe()

In [ ]:
## Matrice de variance covariance pour reprérer les corrélations 
col = ['POPULATION', 'NIVVIE_MEDIAN', 'TAUX_PAUVRETE', 'bistrot_dens', 'age_moyen_bistrots','Nombre candidats','tx_abs']

# Autre table parce que normalisation à venir
df_cor = df_sous_ech

## Normaliser les valeurs 
df_cor[ col ] = (df_sous_ech[col] - df_sous_ech[col].mean()) / df_sous_ech[col].std()

## Matrice de corrélation
cor_matrix = df_cor[col].corr(method='pearson')

display(cor_matrix.round(2)) #pas de grosse corrélation : max entre nb candidats et taux de pauvreté. Faire 2 régressions, une avec, une sans. 


## Déterminants du vote de droite 

On vérifie les effectifs dans chacune des deux modalités

In [ ]:
df_sous_ech['ind1'].value_counts() #effectifs ok dans les deux modalités 

### Régression logistique simple

In [ ]:
# Choix des variables
X = df_sous_ech[['bistrot_dens']]
y = df_sous_ech['ind1']

## Ajouter une constante
X = sm.add_constant(X)

## Fit OLS model
lr = Logit(endog=y,exog=X) 
res = lr.fit(cov_type='HC1') #erreurs robustes  

#résumé des résultats 
print(res.summary()) 

# Coefficient positif : un bistrot de plus est associé à une probabilité plus élevée que la liste élue soit de droite
# Ce n'est pas le sens attendu : il faut contrôler avec d'autres variables. 

### Régression logistique multiple

In [ ]:
# Choix des variables

reg = ['POPULATION', 'TAUX_PAUVRETE','NIVVIE_MEDIAN', 'tx_abs', 'bistrot_dens', 'age_moyen_bistrots'] #'Nombre candidats'

X = df_sous_ech[reg]
y = df_sous_ech['ind1']

## Ajouter une constante
X = sm.add_constant(X)

## Fit OLS model
lr = Logit(endog=y,exog=X) 
res = lr.fit(cov_type='HC1') #erreurs robustes 

#résumé des résultats 
print(res.summary()) 


In [ ]:
# GRAPH IA A SUPPRIMER CAR PAS LE LIEU

import matplotlib.pyplot as plt

plt.figure(figsize=(9, 5))

plt.scatter(
    df_sous_ech.loc[df_sous_ech['ind1'] == 0, 'bistrot_dens'],
    df_sous_ech.loc[df_sous_ech['ind1'] == 0, 'ind1'],
    color='steelblue',
    alpha=0.5,
    s=35,
    label='ind1 = 0'
)

plt.scatter(
    df_sous_ech.loc[df_sous_ech['ind1'] == 1, 'bistrot_dens'],
    df_sous_ech.loc[df_sous_ech['ind1'] == 1, 'ind1'],
    color='red',
    alpha=0.8,
    s=100,
    label='ind1 = 1'
)

plt.xlabel('Densité de bistrots')
plt.ylabel('ind1')
plt.yticks([0, 1])
plt.title('Distribution des observations selon ind1')
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()



## Déterminants du vote RN

On vérifie les effectifs des modalités de la variable cible

In [ ]:
df_sous_ech['ind3'].value_counts() #difficile de mettre beaucoup de variables dans la régression logistique multiple en raison du n faible pour ind3=1

### Régression logistique simple

In [ ]:
# Choix des variables
X = df_sous_ech[['bistrot_dens']]
y = df_sous_ech['ind3']

## Ajouter une constante
X = sm.add_constant(X)

## Fit OLS model
lr = Logit(endog=y,exog=X) 
res = lr.fit() 

#résumé des résultats 
print(res.summary()) # pas significatif 

### Régression logistique multiple 

In [ ]:
# Choix des variables

reg = ['POPULATION', 'TAUX_PAUVRETE','bistrot_dens'] # choix de 3 variables car peu d'effectif 

X = df_sous_ech[reg]
y = df_sous_ech['ind3']

## Ajouter une constante
X = sm.add_constant(X)

## Fit OLS model
lr = Logit(endog=y,exog=X) 
res = lr.fit(cov_type='HC1') #erreurs robustes 

#résumé des résultats 
print(res.summary()) 


In [ ]:
###### GRAPH IA A SUPPRIMER CAR PAS LE LIEUX
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 5))

plt.scatter(
    df_sous_ech.loc[df_sous_ech['ind3'] == 0, 'bistrot_dens'],
    df_sous_ech.loc[df_sous_ech['ind3'] == 0, 'ind3'],
    color='steelblue',
    alpha=0.5,
    s=35,
    label='ind3 = 0'
)

plt.scatter(
    df_sous_ech.loc[df_sous_ech['ind3'] == 1, 'bistrot_dens'],
    df_sous_ech.loc[df_sous_ech['ind3'] == 1, 'ind3'],
    color='red',
    alpha=0.8,
    s=100,
    label='ind3 = 1'
)

plt.xlabel('Densité de bistrots')
plt.ylabel('ind3')
plt.yticks([0, 1])
plt.title('Distribution des observations selon ind3')
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()



### Régression linéaire simple 

In [ ]:
# Choix des variables
X = df_sous_ech[['bistrot_dens']]
y = df_sous_ech['part_votes_ED']

## Ajouter une constante
X = sm.add_constant(X)

## Fit OLS model
ols = sm.OLS(endog=y, exog=X, missing='drop')
res_ols = ols.fit(cov_type='HC1')  # erreurs robustes à l'hétéroscédasticité

# Résumé des résultats
print(res_ols.summary())

# Lecture : part_votes_ED est en points de % des exprimés.
# Le coefficient de bistrot_dens est la variation moyenne (en points de %) de la part des voix
# d'extrême droite au 1er tour associée à une unité de densité de bistrots en plus.

In [ ]:
# Choix des variables
reg = ['POPULATION', 'TAUX_PAUVRETE', 'NIVVIE_MEDIAN', 'tx_abs', 'bistrot_dens', 'age_moyen_bistrots']  # 'Nombre candidats'

X = df_sous_ech[reg]
y = df_sous_ech['part_votes_ED']

## Ajouter une constante
X = sm.add_constant(X)

## Fit OLS model
ols = sm.OLS(endog=y, exog=X, missing='drop')
res_ols = ols.fit(cov_type='HC1')  # erreurs robustes

# Résumé des résultats
print(res_ols.summary())
print(f"Nombre d'observations : {int(res_ols.nobs)}")